# Unit 6 Gen AI Incremental Capstone


A Generative AI-powered advertisement generation system using LLMs and LangChain to create compelling promotional content for BikeEase’s rental services

In [ ]:
# Install packages and import modules
!pip install -q langchain langchain-community

import re
import json
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, pipeline
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline

## Different models to try

- TinyLlama/TinyLlama-1.1B-Chat-v1.0
- google/flan-t5-base
- google/flan-t5-large
- facebook/opt-1.3b
- tiiuae/falcon-rw-1b
- microsoft/phi-2


In [ ]:
# Try using a GPU if available, else use a CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load model/tokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(        # if google-flan-t5 then use AutoModelForSeq2SeqLM
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device).eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Prompt Engineering:

- Generate system rules
- Generate prompt template

In [ ]:
# Instructions for the model to generate the right ad

SYSTEM_MSG = ("""
You are an advertising specialist for BikeEase, a bike rental company.
Your job is to generate marketing ads that are clear, persuasive, and consistent with BikeEase’s brand.

STRICT RULES:
- Use ONLY the provided bike specs, discount, and theme. Do NOT invent prices, emissions numbers, distances, or extra features.
- Do NOT add sections other than the required ones.
- Output MUST match the format exactly. Use the exact labels.
- Voice: Friendly, upbeat, and benefit-driven (focus on what the customer gets).
- Simple language, short sentences.

OUTPUT FORMAT (exactly this, in this order):

[HEADLINE]
(one line, include the user name exactly)

[SUBHEADLINE]
(one sentence, must mention the discount)

[BODY]
(2–4 short sentences, mention exactly 2 features from specs)

[CTA]
(2–6 words)

[HASHTAGS]
(3–6 hashtags, include #BikeEase)


""".strip())

SYSTEM_MSG = """
You are an advertising specialist for BikeEase (bike rental).
Write ONE short ad.

Hard rules:
- Use ONLY the provided bike specs, discount, and theme.
- Do NOT invent prices, engines, brands, ranges, or extra features.
- Output MUST be ONLY what's inside <AD> ... </AD>.
- Must contain exactly these 5 sections in this order:
  [HEADLINE], [SUBHEADLINE], [BODY], [CTA], [HASHTAGS]
- HEADLINE: one line, MUST include the exact user name.
- SUBHEADLINE: one sentence, MUST mention the discount.
- BODY: 2–4 short sentences, mention EXACTLY 2 features from the specs.
- CTA: 2–6 words.
- HASHTAGS: 3–6 hashtags, MUST include #BikeEase.
No extra commentary.
""".strip()

In [ ]:
# Generate prompt template
prompt_tmpl = PromptTemplate(
    input_variables=["bike_specs", "discount", "theme", "channel", "tone", "user_name"],
    template=(
        "Campaign settings:\n"
        "- Channel: {channel}\n"
        "- Tone: {tone}\n\n"
        "Inputs:\n"
        "- User name: {user_name}\n"
        "- Bike specs: {bike_specs}\n"
        "- Discount/Promo: {discount}\n"
        "- Theme: {theme}\n\n"
        "Return ONLY this format between <AD> and </AD>.\n\n"
        "<AD>\n"
        "[HEADLINE]\n"
        "<write one line here>\n\n"
        "[SUBHEADLINE]\n"
        "<write one sentence here>\n\n"
        "[BODY]\n"
        "<write 2-4 short sentences here>\n\n"
        "[CTA]\n"
        "<write 2-6 words here>\n\n"
        "[HASHTAGS]\n"
        "<write 3-6 hashtags here>\n"
        "</AD>\n"
        "[END]"
    )
)

## Build Chat

- Helper Functions:
  - _build_chat_prompt: integrates user-provided content and systems rules
  - _generate_text: preprocesse and tokenize prompt and generates (decodes) new text from the prompt. Adjust hyperparameters accordingly. **Note: T5 models do not require slicing
  - Ad Checker functions:
    - _has_required_labels - checks that required lables are present
    - _name_in_headline - ensures the name of the user is in the headline
    - _validate_ad  - function to check ad
- Ad generation function:
  - generate_ad : takes user variables and generate a prompt, tokenizes and verifies the best version is taken

In [ ]:

# Helper Functions for ad generation

def _build_chat_prompt(user_content: str) -> str:

    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": user_content}
    ]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def _generate_text(prompt: str) -> str:
    # Tokenize prompt (words --> numeric IDs)
    # return_tensors="pt" (PyTorch)/"tf" (TensorFlow)
    inputs = tok(prompt, return_tensors="pt").to(device) # .to(device) = move tensors to GPU if available

    with torch.no_grad():            # No training so no gradient tracking needed
        # Generate new text from the model based on the prompt
        out = model.generate(
            **inputs,                # The tokenized prompt
            max_new_tokens=260,      # Cap on how many new words to generate
            do_sample=True,          # Enable sampling for variety, if False, replace temperature variable with num_beams=3,
            temperature=0.3,         # Randomness: lower = focused, higher = creative
            top_p=0.9,               # Sample from top 90% of probable words
            repetition_penalty=1.15,  # Avoid repeating the same phrase
            no_repeat_ngram_size=4,   # Block repeating any 4-word sequence
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.pad_token_id,
        )

    # For causal models (TinyLlama), slice off prompt tokens, for T5 remove slicing
    gen_ids = out[0][inputs["input_ids"].shape[1]:]

    # Decode the token IDs back into readable text
    return tok.decode(gen_ids, skip_special_tokens=True).strip() # For T5: return tok.decode(out[0], skip_special_tokens=True).strip()

# Ad checker functions

# Define required labels globally for validation
required_labels = ["[HEADLINE]", "[SUBHEADLINE]", "[BODY]", "[CTA]", "[HASHTAGS]"]

if tok.pad_token_id is None and tok.eos_token_id is not None:
    tok.pad_token = tok.eos_token

# Normalize labels text
def normalize_labels(text: str) -> str:
    t = text
    # Convert bare labels to bracketed labels
    t = re.sub(r"^\s*Subheadline\s*$", "[SUBHEADLINE]", t, flags=re.M|re.I)
    t = re.sub(r"^\s*Headline\s*$", "[HEADLINE]", t, flags=re.M|re.I)
    t = re.sub(r"^\s*Body\s*$", "[BODY]", t, flags=re.M|re.I)
    t = re.sub(r"^\s*CTA\s*$", "[CTA]", t, flags=re.M|re.I)
    t = re.sub(r"^\s*Hashtags\s*$", "[HASHTAGS]", t, flags=re.M|re.I)

    # Also handle bracketed-but-wrong-case
    t = re.sub(r"\[(headline)\]", "[HEADLINE]", t, flags=re.I)
    t = re.sub(r"\[(subheadline)\]", "[SUBHEADLINE]", t, flags=re.I)
    t = re.sub(r"\[(body)\]", "[BODY]", t, flags=re.I)
    t = re.sub(r"\[(cta)\]", "[CTA]", t, flags=re.I)
    t = re.sub(r"\[(hashtags)\]", "[HASHTAGS]", t, flags=re.I)

    return t.strip()

# Check if ad has required labels (defined above)
def _has_required_labels(text: str) -> bool:
    t = _normalize_labels(text)
    return all(lbl in t for lbl in required_labels)

# Check if user name is in the ad (to increase personal tone)
def _name_in_headline(text: str, user_name: str) -> bool:
    t = _normalize_labels(text)
    m = re.search(r"\[HEADLINE\]\s*(.+)", t)
    return bool(m) and (user_name.lower() in m.group(1).lower())

# Combine all checkers into a validation function
def validate_ad(text: str, user_name: str) -> dict:
    t = _normalize_labels(text)
    checks = {
        "has_labels": _has_required_labels(t),
        "name_in_headline": _name_in_headline(t, user_name),
        "has_bikeease_hashtag": re.search(r"#bikeease\b", t, flags=re.I) is not None
    }
    checks["score"] = sum(checks.values())
    return checks


def extract_ad_block(text: str) -> str:
    text = text.strip()
    m = re.search(r"<AD>\s*(.*?)\s*</AD>", text, flags=re.S|re.I)
    if m:
        return m.group(1).strip()

    # fallback: cut from first label
    m2 = re.search(r"(\[HEADLINE\].*)", text, flags=re.S)
    return m2.group(1).strip() if m2 else text


In [ ]:
def generate_ad(bike_specs, discount, theme, user_name, channel="Instagram", tone="friendly", retries=4):

    '''The function builds the text prompt that will be fed to the model using the system message (brand rules) and user input:'''

    # Build the user content
    user_content = prompt_tmpl.format(
        user_name=user_name,
        bike_specs=bike_specs,
        discount=discount,
        theme=theme,
        channel=channel,
        tone=tone
    )
    prompt = _build_chat_prompt(user_content)

    best = None
    for _ in range(retries):
        raw = _generate_text(prompt)
        # HARD STOP CUT
        if "[END]" in raw:
            raw = raw.split("[END]")[0] + "[END]"

        raw = extract_ad_block(raw)
        text = normalize_labels(raw)
        checks = validate_ad(text, user_name)

        cand = {"text": text, "checks": checks}
        if best is None or cand["checks"]["score"] > best["checks"]["score"]:
            best = cand
        if checks["score"] == 3:  # perfect score if all rules are verified
            break

    return best


## Test Ad generator

In [ ]:
# Define some user inputs here. You should modify these...
bike_specs = "Electric city bike; pedal assist; up to 70km range; integrated lights; helmet included"
discount = "20% off this week for online bookings"
theme = "Eco-friendly commuting"
user = "Nina"

ad = generate_ad(bike_specs, discount, theme, user, "Instagram", "energetic")
print(ad["text"])


[HEADLINE]
Nina's eco-friendliness is our top priority!
[SUBHEADLINE]
Discover the perfect electric city bike for your next adventure with up to 100 km of range and a sleek design that won't break the bank. Plus, get 20 percent off when you book online today.
[BODY]
With its powerful motor and lightweight frame, the electric city bicle is perfect for commutes around town or exploring new trails. And with its integrated lights and helmet included, you can ride safely and comfortably at night.
[CTA]
Book now and save big on this stylish and sustainable option. Just use promo code "ninabike" during checkout to receive your discount. Don't miss out on this limited-time offer!
[HASHTAGS]
#NinaBikes #ElectricCityBike #GreenTransportation #EcoFriendlyCommuting #SaveMoney #DiscountedRide

[End]


In [ ]:
bike_specs = "Full-suspension mountain bikes"
discount = "50% off this week for online bookings"
theme = "Enjoying-nature"
user = "Bobby"

ad2 = generate_ad(bike_specs, discount, theme, user, "Instagram", "energetic")
print(ad2["text"])

[HEADLINE]
Full-suspend Mountain Bikes | Get 50%% Off This Week!
[SUBHEADLINE]
Enjoy nature with our full-susponed mountain bikes at a great price!
[BODY]
Our range of mountain bikes is designed to provide you with the perfect balance of comfort and performance. Our full-sussion models feature lightweight frames, powerful suspensions, and durable components that will keep you on your feet all day long. Whether you're exploring new trails or simply cruising through town, our bikes have got you covered.

Call Now
Book your next adventure today by calling us at (123) 456-7890 or visiting our website at www.bikerentals.com. Don't miss out on this limited-time offer - get 50 percent off your first booking when you use promo code "BIKEEASE" during checkout.
[CTA]
Book now and save big! Click here to reserve your spot.
[HASHTAGS]
#BikeEasespecialoffer #MountainBiking #FullSuspension #DiscountedPr


# App that generates a pop up ad
I used the script as the backbone of a web app that generates the ad with visuals according to used input. The app uses Tiny-Llama as the core model.

** This part was done with the help of AI (Codex)

## **ReadMe**
## Ad Generator App (TinyLlama)

### This app is based on an ad generating project for Bike Ease, a bike rental company
### App Backbone:
- TinyLlama generation
- PromptTemplate-driven user input formatting
- Validation and retry scoring
- PNG ad visual generation from labeled sections

### Run in command line

```bash
cd "/Users/ns_workstation23/Documents/New project" # Change to your path
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
streamlit run app.py
```

### Files:

- `core/ad_generator.py`: model load, prompt template, chat prompt build, generation, normalization, validation, retries
- `core/visuals.py`: PNG rendering function based on generated sections
- `app.py`: app UI, inputs, generation trigger, preview, downloads

### Notes

- First run can take time while the model downloads.
- If you do not have GPU support, choose `cpu` in the sidebar.
- You can customize both system message and prompt template in the sidebar for any company/campaign. Other customizations include retries (number of versions generated for each ad), font size, ad layout, and ad size (left sidebar)
- Keep these placeholders in the prompt template: `{user_name}`, `{product_specs}`, `{discount}`, `{theme}`, `{channel}`, `{tone}`.
